In [1]:
from osgeo import gdal
import numpy as np
import pandas as pd
#import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../Functions')
import TiffTools as tt

%load_ext autoreload
%autoreload 2

In [ ]:
tt.micmacExport('/Users/chanagan/Downloads/CarsonNevada04142026/M57_NW_Ortho_v2.tif', 
                outname='/Users/chanagan/Downloads/CarsonNevada04142026/mmM57_NW_Ortho_v2.tif', 
                srs='EPSG:32611', outres=[0.6,-0.6], interp=None, a_ullr=None,cutlineDSName=None,nodata=0)

Computing Gray from RGB values
Writing to /Users/chanagan/Downloads/CarsonNevada04142026/mmM57_NW_Ortho_v2.tif


In [ ]:
mm3d Mm2dPosSism 24APR06185248-P1BS-200012802885_01_P007_toAOI.tif \
    26MAY10190720-P1BS-200012802831_01_P001_toAOI.tif \
    CorMin=0.1 Dequant=false DirMEC='MEC_WV/' SsResolOpt=1

In [19]:
folder = '/home/chanagan/tmp/rotationTest/MEC/'
prefiles = ['/home/chanagan/tmp/rotationTest/NCALM2007_DropBox_translate_pos80cm_y_rotated_05292026.tif',
            '/home/chanagan/tmp/rotationTest/USGS_FEMA_2017_D18_merge_clipped_epsg6339_rotated.tif']

In [20]:
tt.micmacPostProcessing(folder=folder,
                         prefiles=prefiles,
                         outprefix=folder)

Nodata value for mask: -9999.0
Setting nodata value to -9999
Saving /home/chanagan/tmp/rotationTest/MEC/NSmicmac.tif
Saving /home/chanagan/tmp/rotationTest/MEC/EWmicmac.tif
Saving /home/chanagan/tmp/rotationTest/MEC/Correlmicmac.tif


In [6]:
par, perp = tt.projectDisp(folder+'MEC100_50/EWmicmac.tif',folder+'MEC100_50/NSmicmac.tif',315,partif=folder+'MEC100_50/ParallelDisp.tif',perptif=folder+'MEC100_50/PerpendicularDisp.tif')

In [5]:
tt.createMicmacParamFile('mm09AUG04191333.tif','mm13MAR11192237.tif',results_directory='MEC100_50/',SzW=100,CorrelMin=0.1,SzW_base=50)

'./ param_LeChantier_Compl.xml written.'

# Raster Rotation


In [7]:
import rasterio
from scipy import ndimage

from osgeo import gdal
import numpy as np
import pandas as pd
#import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../Functions')
import TiffTools as tt

%load_ext autoreload
%autoreload 2

In [54]:
def rotate_raster_centered(arr, gt, angle):
    from scipy import ndimage
    import numpy as np

    # rotate image
    rotated = ndimage.rotate(
        arr,
        angle,
        reshape=True,
        order=0,
        mode="constant",
        cval=-9999
    )
    rotated[rotated < -9000] = -9999
    # original center in pixel coords
    h, w = arr.shape[-2:]
    cx, cy = w / 2, h / 2

    def pixel_to_map(gt, x, y):
        return (
            gt[0] + x * gt[1] + y * gt[2],
            gt[3] + x * gt[4] + y * gt[5]
        )

    map_cx, map_cy = pixel_to_map(gt, cx, cy)

    # rotated shape center
    H, W = rotated.shape[-2:]
    rcx, rcy = W / 2, H / 2

    px_w = gt[1]
    px_h = gt[5]

    # anchor rotated center to same map location
    new_x0 = map_cx - rcx * px_w
    new_y0 = map_cy - rcy * px_h

    new_gt = (new_x0, px_w, 0, new_y0, 0, px_h)

    return rotated, new_gt

In [60]:
files = [
    '/Volumes/ProjectsGarlockCSAF/CSAF_Lidar/RotationTest/MEC/EWmicmac.tif',
    '/Volumes/ProjectsGarlockCSAF/CSAF_Lidar/RotationTest/MEC/NSmicmac.tif'
]
angle = 41
par, perp = tt.projectDisp(files[0],files[1],angle,partif=files[1][:-4]+'_actual.tif',perptif=files[0][:-4]+'_actual.tif')

for file in files:
    im = gdal.Open(file[:-4]+'_actual.tif')
    gt = im.GetGeoTransform()
    imdata = im.GetRasterBand(1).ReadAsArray()
    rotated_data, new_gt = rotate_raster_centered(imdata, gt, angle)

    tt.save_geotiff(
        rotated_data,
        f"{file[:-4]}_rotated.tif",
        new_gt,
        proj,
        nodata=-9999
    )

par, perp = tt.projectDisp(f'{files[0][:-4]}_rotated.tif',
                           f'{files[1][:-4]}_rotated.tif',
                           -1*45,
                           partif='/Volumes/ProjectsGarlockCSAF/CSAF_Lidar/RotationTest/MEC/ParallelDisp315.tif',
                           perptif='/Volumes/ProjectsGarlockCSAF/CSAF_Lidar/RotationTest/MEC/PerpendicularDisp315.tif')



for file in []:
    im = gdal.Open(file)
    gt = im.GetGeoTransform()
    imdata = im.GetRasterBand(1).ReadAsArray()
    rotated_data, new_gt = rotate_raster_centered(imdata, gt, angle)

    tt.save_geotiff(
        rotated_data,
        f"{file[:-4]}_rotated.tif",
        new_gt,
        proj,
        nodata=-9999
    )